# אימון ושימוש חוזר במודלים

קודם בודקים מה כבר אומן בפייז 6. מאמנים רק ניסויים חסרים שנבחרו במפורש. ברירת המחדל של Run all אינה מאמנת דבר.

## פתיחה ב־Colab
העלו מחברת זו ל־Colab. בתחילת runtime חדש תא ההכנה יבקש את `spno-colab-source.zip` המצורף, אלא אם הקוד המעודכן כבר נמצא ב־`PROJECT_ROOT`. גם runtime עם חבילת קוד ישנה יבקש את החבילה המעודכנת. חיבור Drive נעשה דרך ממשק Colab הרגיל.

הגדירו את נתיב תוצרי פייז 6. הנתיב המקורי יכול להיות ב־Drive או בדיסק המקומי של הריצה שטרם נסגרה. אפשר להעתיק את התוצרים ל־Drive באמצעות `COPY_TO`; המקור אינו נמחק. חשוב להשלים את ההעתקה לפני סגירת runtime שבו התוצרים נמצאים רק תחת `/content`.

המחברת קוראת משקולות קיימות. `B-post` הוא A עם projection; אין לו אימון נפרד. checkpoints ישנים אינם מכילים optimizer ולכן אינם נקודת המשך מדויקת לאימון שנקטע. כל אימון חדש דרך הקטלוג שומר התקדמות בסוף כל epoch.


**איתור תוצרים:** ברירת המחדל `SOURCE_ROOT=None` מחפשת תיקייה מלאה בנתיבי Phase 6 המקומיים וב־MyDrive.
אם נמצאה רק חבילת `phase6-standalone-artifacts-*.zip` מלאה, היא נחלצת לתיקייה מקומית נפרדת.
אפשר גם להגדיר `SOURCE_ROOT` ישירות ל־ZIP או לתיקייה שמכילה את התוצרים בתוכה.
אם נמצאו כמה ריצות מלאות, מוצגים הנתיבים לבחירה מפורשת. קובץ הקוד `spno-colab-source.zip` אינו מכיל משקולות או דאטה.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
PROJECT_ROOT = Path(os.environ.get("SPNO_PROJECT_ROOT", "/content/spno-colab" if IN_COLAB else str(Path.cwd())))
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    artifact_code = PROJECT_ROOT / "src/spno/artifacts.py"
    needs_source_update = (not artifact_code.is_file()
                           or "def resolve_standalone_source(" not in artifact_code.read_text())
    if needs_source_update:
        # Upload the updated source bundle, including artifact discovery.
        import zipfile
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload the single spno-colab-source.zip bundle")
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                if not (PROJECT_ROOT / member.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive member")
            archive.extractall(PROJECT_ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT) + "[dirichlet,notebooks]"])
if not (PROJECT_ROOT / "src/spno/workflow.py").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the updated spno source directory")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
OPTIONS = json.loads(os.environ.get("SPNO_OPTIONS", "{}"))
from IPython.display import display, HTML, Image


In [ ]:
# None מחפש אוטומטית. אפשר לציין תיקייה שחולצה, תיקיית האב שלה או ZIP מלא.
# לדוגמה: Path("/content/pin/spno/results/phase6-standalone-artifacts")
SOURCE_ROOT = Path(os.environ["SPNO_SOURCE_ROOT"]) if os.environ.get("SPNO_SOURCE_ROOT") else None
SOURCE_SEARCH_ROOTS = OPTIONS.get("source_search_roots", [
    str(PROJECT_ROOT / "results"), "/content/pin/spno/results", "/content/drive/MyDrive",
])
EXTRACTION_ROOT = Path("/content/spno-phase6-imports") if IN_COLAB else PROJECT_ROOT / "results/phase6-imports"
SOURCE_SEARCH_ROOTS.append(str(EXTRACTION_ROOT))
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/workflow" if IN_COLAB else str(PROJECT_ROOT / "results/colab-workflow")))
CACHE_ROOT = Path("/content/spno-data-cache") if IN_COLAB else None
# אם המקור עדיין בדיסק הזמני של הריצה הישנה: שנו את SOURCE_ROOT לנתיבו.
# COPY_TO מעתיק ל-Drive ובודק שלמות; None משאיר את המקור במקומו.
COPY_TO = None
# קובץ אופציונלי עם {"data": {...}, "train": {...}} מהאימון המקורי.
SOURCE_CONFIG = os.environ.get("SPNO_SOURCE_CONFIG")
# אפשר להצהיר כאן על TrainConfig המקורי אם הדוח אינו מכיל את תקציב האימון.
# לדוגמה, רק אם זו אכן הפקודה שרצה: {"epochs": 80, "batch_size": 256, "learning_rate": 1e-3, "patience": 8}
SOURCE_TRAIN_CONFIG = None
SEEDS = OPTIONS.get("seeds")  # None = כל ה-seeds שהתגלו בפייז 6
DEVICE = OPTIONS.get("device", "auto")
ALLOW_BUDGET_BOUND = OPTIONS.get("allow_budget_bound", False)

# שינוי הניסוי החדש בלבד; אינו משנה את הצהרת פרוטוקול המקור.
TRAIN_OVERRIDES = OPTIONS.get("train_overrides", {})


In [ ]:
from dataclasses import asdict, replace
# Refresh this module when an older bundle was already imported in this runtime.
import spno.artifacts as artifact_tools
importlib.reload(artifact_tools)
from spno.artifacts import atomic_json, copy_standalone, resolve_standalone_source
from spno.config import DataConfig
from spno.train import TrainConfig
from spno.workflow import Workflow
from spno.experiments import pick_device
from spno.phase_workflow import evaluate_phase

print("Locating full Phase 6 artifacts...")
SOURCE_ROOT = resolve_standalone_source(
    SOURCE_ROOT, search_roots=SOURCE_SEARCH_ROOTS, extraction_root=EXTRACTION_ROOT,
)
print("Resolved source:", SOURCE_ROOT)
if COPY_TO is not None:
    SOURCE_ROOT = copy_standalone(SOURCE_ROOT, Path(COPY_TO))
source_config = json.loads(Path(SOURCE_CONFIG).read_text()) if SOURCE_CONFIG else {}
if SOURCE_TRAIN_CONFIG is not None:
    source_config["train"] = SOURCE_TRAIN_CONFIG
workflow = Workflow(
    SOURCE_ROOT, OUTPUT_ROOT, cache_root=CACHE_ROOT,
    data_config=DataConfig(**source_config["data"]) if "data" in source_config else None,
    train_config=TrainConfig(**source_config["train"]) if "train" in source_config else None,
)
SEEDS = workflow.seeds if SEEDS is None else SEEDS
DEVICE = pick_device(DEVICE)
if workflow.train_config is not None:
    atomic_json(OUTPUT_ROOT / "source-config.json", {
        "data": asdict(workflow.data_config), "train": asdict(workflow.train_config),
    })
import html
rows = workflow.inventory()
headers = ["model", "seed", "converged", "best epoch", "history", "protocol", "architecture"]
body = []
for row in rows:
    values = [row["name"], row["seed"], row["converged"], row["metadata"]["best_epoch"],
              "available" if row["history"] else "not recorded", row["protocol_source"] or "UNKNOWN",
              row["metadata"]["architecture"]]
    body.append("<tr>" + "".join("<td>" + html.escape(str(v)) + "</td>" for v in values) + "</tr>")
display(HTML("<table><tr>" + "".join("<th>" + h + "</th>" for h in headers) + "</tr>" + "".join(body) + "</table>"))
print("Seeds:", SEEDS, "Device:", DEVICE)
print("Original training protocol:", workflow.train_config or "UNKNOWN — complete SOURCE_TRAIN_CONFIG before preparing matched experiments")

if TRAIN_OVERRIDES and workflow.train_config is None:
    raise ValueError("Declare the original training protocol before choosing overrides")
REQUESTED_TRAIN_CONFIG = replace(workflow.train_config, **TRAIN_OVERRIDES) if TRAIN_OVERRIDES else None


## בחירת ניסויים להכנה
`TRAIN_PHASES=[]` משאיר את האימון כבוי. הוסיפו 7, 8 או 9 להכנת רשימת ניסויים. פייז 9 עשוי ליצור דאטה חדש: התחילו במספר קטן של ערכי הפרעה. הגדרות הבדיקות אינן חלק מזהות האימון. שינוי פרוטוקול האימון יוצר ניסויים חדשים; המקור נשמר.

פייז 7 מכין את A ו־C1 לכל ערכי `LAMBDAS`, כולל ביקורות lambda=0, וביקורת B-loop יחידה ללא residual. C1+PDE מאומן מאותו אתחול לפי seed כמו C1; לא עושים fine-tuning למודל המקור. שלושה seeds עם ברירת המחדל יוצרים 33 ניסויים, מהם 9 ביקורות שניתן להשתמש בהן מחדש אם הן תואמות. סטטוס כל ניסוי כולל את משקל ה־PDE כדי לזהות מה חסר.


In [ ]:
TRAIN_PHASES = OPTIONS.get("train_phases", [])
LAMBDAS = OPTIONS.get("lambdas", [0.0, 0.01, 0.1, 1.0, 10.0])
FRACTIONS = OPTIONS.get("fractions", [0.05, 0.1, 0.25, 0.5, 1.0])
SIGMAS = OPTIONS.get("sigmas", [0.0, 0.1])
GAMMAS = OPTIONS.get("gammas", [0.0, 0.001])
# None שומר את הפרוטוקול המקורי. לשינוי ניסוי: replace(workflow.train_config, ...).
NEW_TRAIN_CONFIG = REQUESTED_TRAIN_CONFIG
jobs = []
for phase in TRAIN_PHASES:
    jobs.extend(workflow.prepare(phase, seeds=SEEDS, lambdas=LAMBDAS, fractions=FRACTIONS,
                                sigmas=SIGMAS, gammas=GAMMAS, train_config=NEW_TRAIN_CONFIG))
jobs = list({job.identifier: job for job in jobs}.values())
for row in workflow.status(jobs):
    print(json.dumps(row, ensure_ascii=False))
if not jobs:
    print("No training requested. Imported checkpoints remain available.")


## אימון מפורש בלבד
העתיקו ל־`TRAIN_SELECTED_IDS` את מזהי הניסויים שתרצו לאמן. `ready` נטען מחדש; `interrupted` ממשיך מה־epoch האחרון שנשמר; `missing` מתחיל מאתחול לפי seed. `incompatible` מחייב לבדוק את המקור. תוצאה budget-bound נשמרת ואינה מאומנת שוב אוטומטית.

In [ ]:
TRAIN_SELECTED_IDS = OPTIONS.get("train_selected_ids", [])
unknown = set(TRAIN_SELECTED_IDS) - {job.identifier for job in jobs}
if unknown:
    raise ValueError(f"Unknown experiment IDs: {sorted(unknown)}")
selected = [job for job in jobs if job.identifier in TRAIN_SELECTED_IDS]
training_records = workflow.train(selected, device=DEVICE)
print(f"Selected {len(selected)} experiments; all other models were left untouched.")


## בדיקה חוזרת של פייז 6 — ללא אימון
התא כבוי כברירת מחדל משום שגם הבדיקות עצמן עשויות להיות ארוכות. אפשר לבחור רק חלק מזרועות G. פייז 6 שומר את שער ההתכנסות הקיים; `quick` נשאר בדיקת צנרת בלבד.

In [ ]:
RUN_PHASE6_EVALUATION = OPTIONS.get("run_phase6_evaluation", False)
PHASE6_ARMS = OPTIONS.get("arms", ["G5a", "G5b"])
if RUN_PHASE6_EVALUATION:
    result = evaluate_phase(workflow, 6, seeds=SEEDS, device=DEVICE, arms=PHASE6_ARMS)
    print(result["output"])


## הצעד הבא
פתחו את מחברת 7, 8 או 9 עם אותם נתיבי מקור ופלט. כולן טוענות checkpoints בלבד. היסטוריית האימון המקורית, אם נשמרה, זמינה ב־`workflow.inventory()`; היסטוריות חדשות נמצאות לצד המשקולות תחת `experiments/<id>/record.json`.